In [1]:
import pandas as pd
import numpy as np
import joblib
from sklearn.base import clone
from sklearn.model_selection import cross_val_predict, StratifiedKFold

feat = pd.read_csv("../data/telco_features.csv")
clean = pd.read_csv("../data/telco_clean.csv")

X = feat.drop(columns=["Churn"])
y = feat["Churn"]

best_model = joblib.load("../models/churn_model.joblib")

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
proba = cross_val_predict(clone(best_model), X, y, cv=cv,
                          method="predict_proba")[:, 1]

scored = clean.copy()
scored["tenure_group"] = feat["tenure_group"]
scored["churn_probability"] = proba.round(3)
scored["risk_category"] = pd.cut(scored["churn_probability"],
                                 bins=[-0.01, 0.30, 0.60, 1.0],
                                 labels=["Low", "Medium", "High"])

print(scored["risk_category"].value_counts())
print()
print("Actual churn rate in each risk group (%):")
print(scored.groupby("risk_category", observed=True)["Churn"].mean().mul(100).round(1))

scored.to_csv("../data/telco_scored.csv", index=False)
print()
print("Saved telco_scored.csv", scored.shape)

risk_category
Low       4377
Medium    1697
High       969
Name: count, dtype: int64

Actual churn rate in each risk group (%):
risk_category
Low        9.8
Medium    44.1
High      71.3
Name: Churn, dtype: float64

Saved telco_scored.csv (7043, 24)


In [ ]:
segments = ["Contract", "tenure_group", "InternetService", "PaymentMethod",
            "OnlineSecurity", "TechSupport", "SeniorCitizen"]

rows = []
for col in segments:
    g = scored.groupby(col, observed=True)["Churn"].agg(["count", "mean"])
    for value, r in g.iterrows():
        rows.append({"Feature": col, "Group": value,
                     "Customers": int(r["count"]),
                     "Churn rate %": round(r["mean"] * 100, 1)})

seg = pd.DataFrame(rows).sort_values("Churn rate %", ascending=False)
print(seg.head(10).to_string(index=False))

combo = scored[(scored["Contract"] == "Month-to-month") &
               (scored["tenure_group"] == "0-12") &
               (scored["InternetService"] == "Fiber optic")]
print()
print("Month-to-month + first year + fiber optic:")
print("  Customers:", len(combo))
print("  Churn rate %:", round(combo["Churn"].mean() * 100, 1))

        Feature            Group  Customers  Churn rate %
   tenure_group             0-12       2186          47.4
  PaymentMethod Electronic check       2365          45.3
       Contract   Month-to-month       3875          42.7
InternetService      Fiber optic       3096          41.9
 OnlineSecurity               No       3498          41.8
  SeniorCitizen                1       1142          41.7
    TechSupport               No       3473          41.6
   tenure_group            13-24       1024          28.7
  SeniorCitizen                0       5901          23.6
   tenure_group            25-48       1594          20.4

Month-to-month + first year + fiber optic:
  Customers: 916
  Churn rate %: 70.2


: 

## Business insights

### Who churns most (from our data)
- First 12 months: 47.4% churn (2,186 customers)
- Electronic check payers: 45.3%
- Month-to-month contracts: 42.7% (one-year 11.3%, two-year 2.8%)
- Fiber optic internet: 41.9% (DSL 19.0%)
- No online security: 41.8%. No tech support: 41.6%.
- Senior citizens: 41.7% (non-seniors 23.6%)
- New + month-to-month + fiber optic: 916 customers (13% of base), 70.2% churn

### Who to prioritize
- High-risk customers (probability above 60%): 969 customers, 71.3% actually churned
- Medium-risk customers (30% to 60%): 1,697 customers, 44.1% actually churned
- Together they are 38% of customers but about 77% of all churners

### Ideas to test (not proven by this data)
1. Onboarding support in the first 12 months, when churn is highest
2. Incentives to move month-to-month customers onto longer contracts
3. A free trial of online security or tech support for customers without them
4. Encourage automatic payment instead of electronic check
5. Look into fiber optic price and service quality, since its churn is much higher than DSL
6. Contact the High group first, and use low-cost nudges for the Medium group

### What this data cannot tell us
- WHY customers left (there is no complaints or exit-reason column)
- Whether any action above would work (needs an A/B test)
- The cost of offers, so the best cutoff is a business decision
- The model predicts who is likely to leave, not who would respond to an offer